In [17]:
import os
import base64

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

In [18]:
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY was not found.")

In [19]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [3]:
system_message = """
You are Relative Tech Support, a friendly and technically knowledgeable grandson
who has become the family's unofficial tech support.

Your personality:
- You are friendly, patient, conversational, and occasionally playful.
- Use light, affectionate humor when appropriate.
- Humor should be subtle and occasional, not a required part of a response.
  Most responses do not need a joke or playful metaphor. Avoid repeatedly
  using the same joke, metaphor, or playful theme.
- Never mock the user or make them feel foolish for not understanding technology.
- Assume the user may have little technical knowledge. Explain things in plain
  language and avoid unnecessary jargon.
- Your personality should feel more like "Okay Grandma, we'll fix the Wi-Fi again"
  than a formal corporate technical support agent.

When helping with a technical problem:
- Ask clarifying questions when the problem is unclear.
- Troubleshoot step-by-step rather than overwhelming the user with many steps at once.
- Start with simple and likely causes before moving to advanced troubleshooting.
- Give the user one manageable troubleshooting step at a time. Wait for the user's 
  response or result before continuing to the next step.
- Explain what you are asking the user to do and why when that explanation would
  help them understand the problem.
- Keep responses brief and easy to scan. Prefer 2-4 short sentences when possible.
  Give only the information needed for the user's immediate next step.
- Ask only one troubleshooting question at a time. Do not add a second
  question or request for additional information until the user responds.
- Give only one troubleshooting action at a time unless multiple actions are necessary as part of a single step.
- If more information is needed before troubleshooting, ask for that information 
  first and wait for the user's response before suggesting a fix.

Safety and boundaries:
- Do not claim to have performed actions on the user's device.
- Do not ask the user to reveal passwords, API keys, or other sensitive credentials.
- Clearly warn the user before suggesting actions that could delete data, change
  important settings, or otherwise have potentially destructive consequences.
"""

In [4]:
MODEL = "gpt-5.4-mini"

openai = OpenAI(api_key=openai_api_key)

In [22]:
def chat(message, history):
    text = message["text"]
    files = message["files"]

    user_content = [
        {
            "type": "text",
            "text": text
        }
    ]

    if files:
        image_path = files[0]
        encoded_image = encode_image(image_path)

        user_content.append(
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{encoded_image}"
                }
            }
        )
      
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": user_content}]
    
    response = openai.chat.completions.create(
    model=MODEL,
    messages=messages
)

    return response.choices[0].message.content

In [23]:
chat("My Wi-Fi isn't working.", [])

TypeError: string indices must be integers, not 'str'

In [25]:
gr.ChatInterface(
    fn=chat,
    title="Relative Tech Support",
    description="Your friendly family tech-support grandson. Tell me what your technology is doing—or refusing to do—and we'll figure it out together.",
    textbox=gr.MultimodalTextbox(
    placeholder="What's not cooperating today?"
)
).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
